# Study 4 on Google Colab

Runs the locked `docs/PREREGISTRATION_STUDY4.md`. It tests a realistic version of Study 3's fix, a combined pool, and attacks disguised as security questions: 5 versions × 5 seeds = 25 models.

**Before you start**
1. *Runtime → Change runtime type → A100 GPU.* Every step needs the GPU, including step 1.
2. `HF_TOKEN` must be in Colab's 🔑 *Secrets* panel, with notebook access turned on. **Never paste the token into a cell.**
3. The Study 3 folder `MyDrive/study3` must still be in your Drive. Study 4 reuses its matched pool.
4. The project owner's Study 4 approval line must already be in `AGENTS.md`.

**LMSYS licence.** Text stays in `MyDrive/study4/private_lmsys_text_do_not_share`. Never share it. Only `MyDrive/study4/public` comes back.

**Blinding.** Until step 3 finishes, do not open `public/scores/` or the private `logs/`.

**Resuming.** If Colab disconnects, run the setup cells again, then the step you were on. Finished work is skipped.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/study4'
STUDY3 = '/content/drive/MyDrive/study3'

In [ ]:
import os
from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')  # from Secrets; never printed
if not os.path.exists('/content/prompt_injection'):
    os.system('git clone -q -b claude/vigilant-fermi-r2ovyf https://github.com/dannyg26/prompt_injection /content/prompt_injection')
os.chdir('/content/prompt_injection')
!git pull -q origin claude/vigilant-fermi-r2ovyf
!git log --oneline -1

In [ ]:
!pip install -q -r requirements-lock.txt
!pip install -q transformers==4.57.1 datasets==4.4.1 python-dotenv==1.0.1 pandas sentencepiece protobuf
!pip install -q --no-deps -e .
!python -c "import torch, transformers, datasets; print(torch.__version__, transformers.__version__, datasets.__version__)"

## 1. Pools: mines the realistic pool with a released detector; no training (about 1 hour)

In [ ]:
!PYTHONPATH=src python scripts/run_study4.py --out "$OUT" --study3-out "$STUDY3" --stage pools 2>&1 | grep '\[study3'

## 2. Pilot: one model, prints the time

In [ ]:
!PYTHONPATH=src python scripts/run_study4.py --out "$OUT" --study3-out "$STUDY3" --stage pilot 2>&1 | grep '\[study3'

## 3. All remaining models, then the analysis (about 20 hours; resumable)

In [ ]:
!PYTHONPATH=src python scripts/run_study4.py --out "$OUT" --study3-out "$STUDY3" --stage all 2>&1 | grep '\[study3'

## After it finishes

Download these two files and send them back:
- `MyDrive/study4/public/study4_results.json`
- `MyDrive/study4/public/pool_manifest.json`

For the blind audit, label `private_lmsys_text_do_not_share/audit_B1_blind.csv` the same way as Study 3's audit, and send only the counts.

In [ ]:
from google.colab import files
files.download(f'{OUT}/public/study4_results.json')
files.download(f'{OUT}/public/pool_manifest.json')